In [4]:
from datetime import datetime

def get_historical_rainfall(lat, lon, start_date, end_date):
    import requests
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "precipitation,rain,temperature_2m,windspeed_10m",
        "timezone": "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data = response.json()
    df = pd.DataFrame({
        "time": data["hourly"]["time"],
        "precipitation_mm": data["hourly"]["precipitation"],
        "rain_mm": data["hourly"]["rain"],
        "temperature_c": data["hourly"]["temperature_2m"],
        "windspeed_kmh": data["hourly"]["windspeed_10m"]
    })
    df["time"] = pd.to_datetime(df["time"])
    return df

# Get weather for your exact collection window
start = check_df['timestamp'].min().strftime('%Y-%m-%d')
end = check_df['timestamp'].max().strftime('%Y-%m-%d')
print(f"Fetching weather from {start} to {end}...")

rain_df = get_historical_rainfall(13.0827, 80.2707, start, end)
print(f"Weather rows: {rain_df.shape[0]}")
print(rain_df.head())

NameError: name 'check_df' is not defined

In [5]:
# Round traffic timestamps to nearest hour for merging
check_df['timestamp_hour'] = check_df['timestamp'].dt.floor('h')

# Merge weather into traffic data
merged_df = pd.merge_asof(
    check_df.sort_values('timestamp'),
    rain_df.rename(columns={'time': 'timestamp_hour'}).sort_values('timestamp_hour'),
    on='timestamp_hour',
    direction='nearest'
)

# Add monsoon flag (Chennai NE monsoon: Oct-Dec, SW monsoon: Jun-Sep)
merged_df['is_monsoon'] = merged_df['timestamp'].dt.month.isin([6, 7, 8, 9, 10, 11, 12])

# Add time features
merged_df['hour'] = merged_df['timestamp'].dt.hour
merged_df['day_of_week'] = merged_df['timestamp'].dt.dayofweek
merged_df['is_peak_hour'] = merged_df['hour'].isin([8, 9, 10, 17, 18, 19, 20])

print(f"Merged dataset shape: {merged_df.shape}")
print(f"\nColumns: {merged_df.columns.tolist()}")
print(f"\nSample merged row:")
print(merged_df[['timestamp', 'junction', 'current_speed', 
                  'precipitation_mm', 'temperature_c', 
                  'is_monsoon', 'is_peak_hour']].head(10))

# Save
merged_df.to_csv('../data/raw/chennai_traffic_weather.csv', index=False)
print("\n✅ Saved to data/raw/chennai_traffic_weather.csv")

NameError: name 'check_df' is not defined

In [6]:
import torch

print("Loading ITD dataset...")
itd = torch.load('../data/raw/best_xl_ITD_v1.2.pt', map_location='cpu', weights_only=False)

print(f"Type: {type(itd)}")

if isinstance(itd, dict):
    print(f"Keys: {list(itd.keys())}")
elif isinstance(itd, list):
    print(f"Length: {len(itd)}")
    print(f"First element type: {type(itd[0])}")
    print(f"First element: {itd[0]}")
else:
    print(itd)

Loading ITD dataset...
Type: <class 'dict'>
Keys: ['date', 'version', 'license', 'docs', 'epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'train_args', 'train_metrics', 'train_results']


In [7]:
print(f"Epoch trained: {itd['epoch']}")
print(f"Best fitness: {itd['best_fitness']}")
print(f"Date trained: {itd['date']}")
print(f"Version: {itd['version']}")

print(f"\nTrain args:")
for k, v in itd['train_args'].items():
    print(f"  {k}: {v}")

Epoch trained: -1
Best fitness: None
Date trained: 2025-03-11T00:50:23.035205
Version: 8.3.74

Train args:
  task: detect
  mode: train
  model: yolo11x.pt
  data: /home/user1/data/ITD/v1.2_itd/data.yaml
  epochs: 800
  time: None
  patience: 100
  batch: 8
  imgsz: 992
  save: True
  save_period: -1
  cache: disk
  device: 0
  workers: 8
  project: None
  name: ITD_v1.2_xl
  exist_ok: False
  pretrained: True
  optimizer: auto
  verbose: True
  seed: 0
  deterministic: True
  single_cls: False
  rect: False
  cos_lr: False
  close_mosaic: 10
  resume: False
  amp: True
  fraction: 1.0
  profile: False
  freeze: None
  multi_scale: False
  overlap_mask: True
  mask_ratio: 4
  dropout: 0.0
  val: True
  split: val
  save_json: False
  save_hybrid: False
  conf: None
  iou: 0.7
  max_det: 300
  half: False
  dnn: False
  plots: True
  source: None
  vid_stride: 1
  stream_buffer: False
  visualize: False
  augment: False
  agnostic_nms: False
  classes: None
  retina_masks: False
  embed

In [8]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [9]:
from ultralytics import YOLO

# Load the ITD model
model_itd = YOLO('../data/raw/best_xl_ITD_v1.2.pt')

# Check what classes it detects
print("Vehicle classes this model can detect:")
for idx, name in model_itd.names.items():
    print(f"  {idx}: {name}")

Vehicle classes this model can detect:
  0: two wheeler
  1: autorickshaw
  2: car
  3: bus
  4: LCV
  5: truck
  6: bicycle
  7: pedestrain


In [ ]:
pip install mapillary

In [5]:
import requests
import os
from PIL import Image
from io import BytesIO
import pandas as pd
import torch

# Paste your Mapillary token here
MLY_TOKEN = "MLY|27295136363512105|a45ac25d2a589fef4308b128f805afce"

# Your 20 junctions with coordinates
junctions = {
    'Kathipara': (13.0107, 80.2016),
    'Guindy': (13.0067, 80.2206),
    'T_Nagar_Panagal': (13.0418, 80.2341),
    'Anna_Salai_Teynampet': (13.0455, 80.2500),
    'Egmore': (13.0732, 80.2609),
    'Central_Station': (13.0827, 80.2785),
    'Koyambedu': (13.0694, 80.1948),
    'Vadapalani': (13.0531, 80.2121),
    'Ashok_Nagar': (13.0306, 80.2174),
    'Adyar_Signal': (13.0012, 80.2565),
    'Velachery_Junction': (12.9815, 80.2180),
    'Tambaram': (12.9249, 80.1000),
    'Perungudi_OMR': (12.9611, 80.2400),
    'Thoraipakkam_OMR': (12.9388, 80.2333),
    'Porur': (13.0374, 80.1572),
    'Anna_Nagar_Roundtana': (13.0850, 80.2101),
    'Perambur': (13.1102, 80.2456),
    'Mylapore': (13.0339, 80.2676),
    'Chromepet': (12.9516, 80.1462),
    'Kelambakkam_OMR': (12.7967, 80.2206)
}

def get_mapillary_image(lat, lon, token, radius=100):
    """Get nearest Mapillary image to a coordinate"""
    url = "https://graph.mapillary.com/images"
    params = {
        'access_token': token,
        'fields': 'id,thumb_2048_url,captured_at,geometry',
        'bbox': f'{lon-0.001},{lat-0.001},{lon+0.001},{lat+0.001}',
        'limit': 1
    }
    response = requests.get(url, params=params)
    data = response.json()
    if 'data' in data and len(data['data']) > 0:
        return data['data'][0]
    return None

def download_image(url, token):
    """Download image from Mapillary"""
    headers = {'Authorization': f'OAuth {token}'}
    response = requests.get(url, headers=headers)
    return Image.open(BytesIO(response.content))

# Get images and run detection
results_list = []
os.makedirs('../data/raw/mapillary_images', exist_ok=True)

print("Fetching Mapillary images and running ITD detection...")
for junction_name, (lat, lon) in junctions.items():
    print(f"\nProcessing {junction_name}...")
    
    # Get image metadata
    img_data = get_mapillary_image(lat, lon, MLY_TOKEN)
    
    if img_data is None:
        print(f"  No image found for {junction_name}")
        continue
    
    try:
        # Download image
        img_url = img_data['thumb_2048_url']
        img = download_image(img_url, MLY_TOKEN)
        img_path = f'../data/raw/mapillary_images/{junction_name}.jpg'
        img.save(img_path)
        
        # Run ITD detection
        det_results = model_itd(img_path, conf=0.3, verbose=False)
        boxes = det_results[0].boxes
        
        # Count vehicles by class
        counts = {name: 0 for name in model_itd.names.values()}
        for cls_id in boxes.cls.tolist():
            counts[model_itd.names[int(cls_id)]] += 1
        
        # Save annotated image
        det_results[0].save(f'../data/raw/mapillary_images/{junction_name}_detected.jpg')
        
        total = sum(counts.values())
        print(f"  ✅ Detected {total} objects: {counts}")
        
        results_list.append({
            'junction': junction_name,
            'lat': lat,
            'lon': lon,
            'image_id': img_data['id'],
            **counts,
            'total_vehicles': total
        })
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        continue

# Save results
modal_df = pd.DataFrame(results_list)
modal_df.to_csv('../data/raw/junction_vehicle_counts.csv', index=False)
print(f"\n✅ Done! Processed {len(results_list)}/20 junctions")
print(modal_df)

Fetching Mapillary images and running ITD detection...

Processing Kathipara...
  No image found for Kathipara

Processing Guindy...
  ❌ Error: name 'model_itd' is not defined

Processing T_Nagar_Panagal...
  No image found for T_Nagar_Panagal

Processing Anna_Salai_Teynampet...
  No image found for Anna_Salai_Teynampet

Processing Egmore...


ConnectionError: ('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None))

In [2]:
import os
os.makedirs('../data/processed', exist_ok=True)

merged_full.to_csv('../data/processed/chennai_full_dataset.csv', index=False)
print("✅ Saved to processed/chennai_full_dataset.csv")
print(f"Final dataset shape: {merged_full.shape}")

NameError: name 'merged_full' is not defined

In [3]:
# Merge vehicle counts into our traffic+weather dataset
merged_full = pd.merge(
    merged_df,
    modal_df[['junction', 'two wheeler', 'autorickshaw', 'car', 
              'bus', 'LCV', 'truck', 'bicycle', 'pedestrain', 'total_vehicles']],
    on='junction',
    how='left'
)

# Fill missing junctions with 0 for now
vehicle_cols = ['two wheeler', 'autorickshaw', 'car', 'bus', 
                'LCV', 'truck', 'bicycle', 'pedestrain', 'total_vehicles']
merged_full[vehicle_cols] = merged_full[vehicle_cols].fillna(0)

# Add PCU weighted density (Passenger Car Unit - standard Indian traffic metric)
# PCU values: 2W=0.5, Auto=0.8, Car=1.0, Bus=3.0, LCV=1.5, Truck=3.0, Bicycle=0.5
merged_full['pcu_density'] = (
    merged_full['two wheeler'] * 0.5 +
    merged_full['autorickshaw'] * 0.8 +
    merged_full['car'] * 1.0 +
    merged_full['bus'] * 3.0 +
    merged_full['LCV'] * 1.5 +
    merged_full['truck'] * 3.0 +
    merged_full['bicycle'] * 0.5
)

print(f"Final dataset shape: {merged_full.shape}")
print(f"Columns: {merged_full.columns.tolist()}")
print(f"\nSample with PCU density:")
print(merged_full[['junction', 'current_speed', 'precipitation_mm', 
                    'pcu_density', 'is_peak_hour', 'is_monsoon']].head(10))

# Save final dataset
merged_full.to_csv('../data/processed/chennai_full_dataset.csv', index=False)
print("\n✅ Saved to processed/chennai_full_dataset.csv")

NameError: name 'pd' is not defined

In [1]:
print(f"Dataset shape: {merged_full.shape}")
print(f"\nFeature summary:")
print(merged_full[['current_speed', 'precipitation_mm', 'temperature_c', 
                    'pcu_density', 'is_peak_hour', 'is_monsoon', 
                    'hour', 'day_of_week']].describe())
print(f"\nMissing values:")
print(merged_full.isnull().sum())

NameError: name 'merged_full' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import numpy as np

# ── 1. Build graph from your 20 junctions ──────────────────────────────────
junction_list = merged_full['junction'].unique().tolist()
junction_to_idx = {j: i for i, j in enumerate(junction_list)}
num_nodes = len(junction_list)
print(f"Number of junction nodes: {num_nodes}")

# Connect junctions that are geographically close (within ~5km)
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Get coordinates per junction
coords = merged_full.groupby('junction')[['lat','lon']].first().to_dict('index')

# Build edges between junctions within 10km of each other
edge_src, edge_dst = [], []
for j1 in junction_list:
    for j2 in junction_list:
        if j1 != j2:
            d = haversine(coords[j1]['lat'], coords[j1]['lon'],
                         coords[j2]['lat'], coords[j2]['lon'])
            if d <= 10.0:
                edge_src.append(junction_to_idx[j1])
                edge_dst.append(junction_to_idx[j2])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
print(f"Number of edges between junctions: {edge_index.shape[1]}")
print(f"✅ Graph structure built!")

In [ ]:
# ── 2. Build feature matrix per snapshot ───────────────────────────────────

feature_cols = [
    'current_speed', 'free_flow_speed',
    'precipitation_mm', 'temperature_c', 'windspeed_kmh',
    'pcu_density', 'two wheeler', 'autorickshaw', 'car',
    'bus', 'LCV', 'truck', 'bicycle',
    'hour', 'day_of_week', 'is_peak_hour', 'is_monsoon'
]

# Convert booleans to int
merged_full['is_peak_hour'] = merged_full['is_peak_hour'].astype(int)
merged_full['is_monsoon'] = merged_full['is_monsoon'].astype(int)
merged_full['road_closure'] = merged_full['road_closure'].astype(int)

# Get unique timestamps (snapshots)
timestamps = sorted(merged_full['timestamp'].unique())
print(f"Total snapshots: {len(timestamps)}")

# Build list of (X, y) pairs — one per snapshot
# X = feature matrix [num_nodes x num_features]
# y = target speeds  [num_nodes x 1]
snapshots_X = []
snapshots_y = []

for ts in timestamps:
    snap = merged_full[merged_full['timestamp'] == ts]
    snap = snap.set_index('junction').reindex(junction_list)
    
    X = snap[feature_cols].fillna(0).values.astype(np.float32)
    y = snap['current_speed'].fillna(0).values.astype(np.float32)
    
    snapshots_X.append(X)
    snapshots_y.append(y)

snapshots_X = np.array(snapshots_X)  # [T, N, F]
snapshots_y = np.array(snapshots_y)  # [T, N]

print(f"Feature tensor shape: {snapshots_X.shape}")
print(f"Target tensor shape:  {snapshots_y.shape}")
print(f"Number of features:   {len(feature_cols)}")
print(f"✅ Snapshots ready!")

In [ ]:
# ── 3. Define the ST-GNN Model ─────────────────────────────────────────────

class MonsoonAwareSTGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        
        # Spatial: Graph Convolution layers
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        
        # Temporal: GRU layer
        self.gru = nn.GRU(hidden_channels, hidden_channels, 
                          batch_first=True, num_layers=2)
        
        # Monsoon-aware attention (learns to weight rain features)
        self.monsoon_attn = nn.Sequential(
            nn.Linear(3, hidden_channels),  # rain, temp, is_monsoon
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.Sigmoid()
        )
        
        # Mixed-modal attention (learns to weight vehicle types)
        self.modal_attn = nn.Sequential(
            nn.Linear(7, hidden_channels),  # 7 vehicle classes
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.Sigmoid()
        )
        
        # Output layer
        self.output = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x_seq, edge_index):
        # x_seq: [T, N, F]
        T, N, F = x_seq.shape
        node_embeds = []
        
        for t in range(T):
            x = x_seq[t]  # [N, F]
            
            # Extract monsoon features (precipitation, temperature, is_monsoon)
            monsoon_feat = x[:, [2, 3, 16]]  # indices in feature_cols
            monsoon_weight = self.monsoon_attn(monsoon_feat)
            
            # Extract modal features (7 vehicle classes)
            modal_feat = x[:, [6, 7, 8, 9, 10, 11, 12]]
            modal_weight = self.modal_attn(modal_feat)
            
            # Graph convolution
            h = F.relu(self.gcn1(x, edge_index))
            h = self.dropout(h)
            h = F.relu(self.gcn2(h, edge_index))
            
            # Apply monsoon and modal attention
            h = h * monsoon_weight * modal_weight
            
            node_embeds.append(h)
        
        # Stack temporal sequence: [N, T, hidden]
        node_embeds = torch.stack(node_embeds, dim=1)
        
        # GRU over time
        gru_out, _ = self.gru(node_embeds)
        
        # Take last timestep output
        out = self.output(gru_out[:, -1, :])  # [N, 1]
        return out.squeeze(-1)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MonsoonAwareSTGNN(
    in_channels=len(feature_cols),
    hidden_channels=64,
    out_channels=1
).to(device)

print(f"\nModel architecture:")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
import torch.nn.functional as F
print("F reloaded:", F)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt

class MonsoonAwareSTGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.gru = nn.GRU(hidden_channels, hidden_channels, 
                          batch_first=True, num_layers=2)
        self.monsoon_attn = nn.Sequential(
            nn.Linear(3, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels), nn.Sigmoid()
        )
        self.modal_attn = nn.Sequential(
            nn.Linear(7, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels), nn.Sigmoid()
        )
        self.output = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x_seq, edge_index):
        T, N, Feat = x_seq.shape
        node_embeds = []
        for t in range(T):
            x = x_seq[t]
            monsoon_feat = x[:, [2, 3, 16]]
            monsoon_weight = self.monsoon_attn(monsoon_feat)
            modal_feat = x[:, [6, 7, 8, 9, 10, 11, 12]]
            modal_weight = self.modal_attn(modal_feat)
            h = F.relu(self.gcn1(x, edge_index))
            h = self.dropout(h)
            h = F.relu(self.gcn2(h, edge_index))
            h = h * monsoon_weight * modal_weight
            node_embeds.append(h)
        node_embeds = torch.stack(node_embeds, dim=1)
        gru_out, _ = self.gru(node_embeds)
        out = self.output(gru_out[:, -1, :])
        return out.squeeze(-1)

# Reinitialize
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MonsoonAwareSTGNN(in_channels=len(feature_cols), 
                           hidden_channels=64, 
                           out_channels=1).to(device)
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

# Training loop
train_losses = []
num_epochs = 100

print(f"Training on: {device}")
print("Training MonsoonAwareSTGNN...")

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train, edge_index_dev)
    loss = F.mse_loss(pred, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test, edge_index_dev)
            test_loss = F.mse_loss(test_pred, y_test)
            test_pred_kmh = test_pred * (speed_max - speed_min) + speed_min
            y_test_kmh = y_test * (speed_max - speed_min) + speed_min
            mae = F.l1_loss(test_pred_kmh, y_test_kmh).item()
        print(f"Epoch {epoch+1:3d} | Train Loss: {loss.item():.4f} | Test Loss: {test_loss.item():.4f} | MAE: {mae:.2f} km/h")

# Plot
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('MonsoonAwareSTGNN Training Curve')
plt.legend()
plt.savefig('../logs/training_curve.png', dpi=150)
plt.show()

# Save
torch.save(model.state_dict(), '../models/monsoon_stgnn.pt')
print("✅ Training complete! Model saved.")

In [ ]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test, edge_index_dev)
    test_pred_kmh = test_pred * (speed_max - speed_min) + speed_min
    y_test_kmh = y_test * (speed_max - speed_min) + speed_min
    
    mae = F.l1_loss(test_pred_kmh, y_test_kmh).item()
    mse = F.mse_loss(test_pred_kmh, y_test_kmh).item()
    rmse = mse ** 0.5
    mape = (torch.abs(test_pred_kmh - y_test_kmh) / (y_test_kmh + 1e-6)).mean().item() * 100

print("=" * 45)
print("   MonsoonAwareSTGNN — Final Evaluation")
print("=" * 45)
print(f"   MAE  : {mae:.2f} km/h")
print(f"   RMSE : {rmse:.2f} km/h")
print(f"   MAPE : {mape:.2f} %")
print("=" * 45)

# Per junction predictions vs actual
print("\nPer-junction predictions (last test snapshot):")
print(f"{'Junction':<25} {'Actual':>8} {'Predicted':>10} {'Error':>8}")
print("-" * 55)
for i, jname in enumerate(junction_list):
    actual = y_test_kmh[-1, i].item()
    predicted = test_pred_kmh[-1, i].item() if test_pred_kmh.dim() > 1 else test_pred_kmh[i].item()
    error = abs(actual - predicted)
    print(f"{jname:<25} {actual:>8.1f} {predicted:>10.1f} {error:>8.1f}")

In [ ]:
# Fix: evaluate on each test snapshot separately
model.eval()
all_mae, all_rmse, all_mape = [], [], []

with torch.no_grad():
    for i in range(X_test.shape[0]):
        x_single = X_test[i:i+1]  # [1, N, F]
        y_single = y_test[i]       # [N]
        
        pred = model(x_single, edge_index_dev)  # [N]
        pred_kmh = pred * (speed_max - speed_min) + speed_min
        y_kmh = y_single * (speed_max - speed_min) + speed_min
        
        mae = F.l1_loss(pred_kmh, y_kmh).item()
        rmse = F.mse_loss(pred_kmh, y_kmh).item() ** 0.5
        mape = (torch.abs(pred_kmh - y_kmh) / (y_kmh + 1e-6)).mean().item() * 100
        
        all_mae.append(mae)
        all_rmse.append(rmse)
        all_mape.append(mape)

print("=" * 45)
print("   MonsoonAwareSTGNN — Corrected Evaluation")
print("=" * 45)
print(f"   MAE  : {np.mean(all_mae):.2f} km/h")
print(f"   RMSE : {np.mean(all_rmse):.2f} km/h")
print(f"   MAPE : {np.mean(all_mape):.2f} %")
print("=" * 45)

# Save results
results = {
    'model': 'MonsoonAwareSTGNN',
    'dataset': 'Chennai Real Data (TomTom + Weather + Mapillary/ITD)',
    'nodes': 20,
    'snapshots': 17,
    'features': len(feature_cols),
    'MAE': round(np.mean(all_mae), 2),
    'RMSE': round(np.mean(all_rmse), 2),
    'MAPE': round(np.mean(all_mape), 2),
    'epochs': 100,
    'parameters': 64385
}

import json
with open('../logs/final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✅ Results saved to logs/final_results.json")
print(json.dumps(results, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('MonsoonAwareSTGNN — Chennai Traffic Prediction Dashboard', 
             fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted speeds (last snapshot)
model.eval()
with torch.no_grad():
    x_last = X_test[-1:] 
    pred_last = model(x_last, edge_index_dev)
    pred_kmh = (pred_last * (speed_max - speed_min) + speed_min).cpu().numpy()
    actual_kmh = (y_test[-1] * (speed_max - speed_min) + speed_min).cpu().numpy()

ax1 = axes[0, 0]
x_pos = np.arange(len(junction_list))
width = 0.35
ax1.bar(x_pos - width/2, actual_kmh, width, label='Actual', color='steelblue', alpha=0.8)
ax1.bar(x_pos + width/2, pred_kmh, width, label='Predicted', color='orange', alpha=0.8)
ax1.set_xticks(x_pos)
ax1.set_xticklabels([j[:8] for j in junction_list], rotation=45, ha='right', fontsize=7)
ax1.set_ylabel('Speed (km/h)')
ax1.set_title('Actual vs Predicted Speed per Junction')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Training loss curve
ax2 = axes[0, 1]
ax2.plot(train_losses, color='green', linewidth=1.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Training Loss Curve')
ax2.grid(alpha=0.3)

# Plot 3: Speed heatmap across junctions over time
ax3 = axes[1, 0]
speed_data = merged_full.pivot_table(
    values='current_speed', 
    index='timestamp', 
    columns='junction', 
    aggfunc='mean'
).fillna(0)
im = ax3.imshow(speed_data.values.T, aspect='auto', cmap='RdYlGn', 
                vmin=0, vmax=60)
ax3.set_yticks(range(len(speed_data.columns)))
ax3.set_yticklabels([c[:10] for c in speed_data.columns], fontsize=6)
ax3.set_xlabel('Time Snapshot')
ax3.set_title('Speed Heatmap (Green=Fast, Red=Slow)')
plt.colorbar(im, ax=ax3, label='km/h')

# Plot 4: Model metrics summary
ax4 = axes[1, 1]
ax4.axis('off')
metrics_text = f"""
Model: MonsoonAwareSTGNN

Architecture:
  • GCN Layer 1: 17 → 64
  • GCN Layer 2: 64 → 64  
  • GRU: 64 → 64 (2 layers)
  • Monsoon Attention Module ✓
  • Mixed-Modal Attention Module ✓

Dataset:
  • City: Chennai, Tamil Nadu
  • Junctions: 20
  • Snapshots: 17
  • Features: 17

Performance:
  • MAE  : 9.06 km/h
  • RMSE : 10.59 km/h
  • MAPE : 39.53 %

Data Sources:
  • TomTom Traffic API (real-time)
  • Open-Meteo Weather API
  • Mapillary Street Images
  • IIT Roorkee ITD v1.2 (YOLO)
  • OpenStreetMap Road Network
"""
ax4.text(0.05, 0.95, metrics_text, transform=ax4.transAxes,
         fontsize=9, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.tight_layout()
plt.savefig('../logs/dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved to logs/dashboard.png")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Get actual speeds in km/h for test set
y_test_actual = (y_test * (speed_max - speed_min) + speed_min).cpu().numpy()
y_train_actual = (y_train * (speed_max - speed_min) + speed_min).cpu().numpy()

# ── Baseline 1: Historical Average ─────────────────────────────────────────
train_mean = y_train_actual.mean(axis=0)  # per junction mean
ha_preds = np.tile(train_mean, (y_test_actual.shape[0], 1))
ha_mae  = mean_absolute_error(y_test_actual.flatten(), ha_preds.flatten())
ha_rmse = mean_squared_error(y_test_actual.flatten(), ha_preds.flatten()) ** 0.5
ha_mape = (np.abs(y_test_actual - ha_preds) / (y_test_actual + 1e-6)).mean() * 100

print(f"Baseline 1 — Historical Average:")
print(f"  MAE : {ha_mae:.2f} km/h")
print(f"  RMSE: {ha_rmse:.2f} km/h")
print(f"  MAPE: {ha_mape:.2f} %")

# ── Baseline 2: Last Value (Persistence) ───────────────────────────────────
last_val = y_train_actual[-1]  # last training snapshot
lv_preds = np.tile(last_val, (y_test_actual.shape[0], 1))
lv_mae  = mean_absolute_error(y_test_actual.flatten(), lv_preds.flatten())
lv_rmse = mean_squared_error(y_test_actual.flatten(), lv_preds.flatten()) ** 0.5
lv_mape = (np.abs(y_test_actual - lv_preds) / (y_test_actual + 1e-6)).mean() * 100

print(f"\nBaseline 2 — Last Value (Persistence):")
print(f"  MAE : {lv_mae:.2f} km/h")
print(f"  RMSE: {lv_rmse:.2f} km/h")
print(f"  MAPE: {lv_mape:.2f} %")

# ── Baseline 3: Linear Regression ──────────────────────────────────────────
X_train_np = snapshots_X[:split].reshape(split, -1)
X_test_np  = snapshots_X[split:].reshape(len(timestamps)-split, -1)
y_train_flat = y_train_actual.reshape(split, -1)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_np)
X_test_sc  = scaler.transform(X_test_np)

lr_model = LinearRegression()
lr_model.fit(X_train_sc, y_train_flat)
lr_preds = lr_model.predict(X_test_sc)

lr_mae  = mean_absolute_error(y_test_actual.flatten(), lr_preds.flatten())
lr_rmse = mean_squared_error(y_test_actual.flatten(), lr_preds.flatten()) ** 0.5
lr_mape = (np.abs(y_test_actual - lr_preds) / (y_test_actual + 1e-6)).mean() * 100

print(f"\nBaseline 3 — Linear Regression:")
print(f"  MAE : {lr_mae:.2f} km/h")
print(f"  RMSE: {lr_rmse:.2f} km/h")
print(f"  MAPE: {lr_mape:.2f} %")

# ── Our Model ──────────────────────────────────────────────────────────────
our_mae  = 9.06
our_rmse = 10.59
our_mape = 39.53

print(f"\nOur Model — MonsoonAwareSTGNN:")
print(f"  MAE : {our_mae:.2f} km/h")
print(f"  RMSE: {our_rmse:.2f} km/h")
print(f"  MAPE: {our_mape:.2f} %")

In [ ]:
with open('../src/data/collector.py', 'r') as f:
    print(f.read())

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'plotly'])
print("Done!")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Reload latest data
df = pd.read_csv('../data/raw/chennai_traffic_log.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

# ── Dashboard Layout ───────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Live Speed by Junction',
        'Speed Over Time — Top 5 Junctions',
        'Junction Map — Chennai',
        'Traffic Congestion Index'
    ),
    specs=[
        [{"type": "bar"}, {"type": "scatter"}],
        [{"type": "scattermapbox"}, {"type": "bar"}]
    ]
)

# ── Plot 1: Latest speed per junction ──────────────────────────────────────
latest = df.groupby('junction').last().reset_index()
colors = ['red' if s < 15 else 'orange' if s < 25 else 'green' 
          for s in latest['current_speed']]

fig.add_trace(go.Bar(
    x=latest['junction'],
    y=latest['current_speed'],
    marker_color=colors,
    name='Current Speed',
    text=latest['current_speed'],
    textposition='outside'
), row=1, col=1)

# ── Plot 2: Speed over time for top 5 busiest junctions ───────────────────
busy_junctions = ['Kathipara', 'T_Nagar_Panagal', 'Koyambedu', 
                  'Anna_Nagar_Roundtana', 'Adyar_Signal']
colors_line = ['blue', 'red', 'green', 'purple', 'orange']

for junc, col in zip(busy_junctions, colors_line):
    jdf = df[df['junction'] == junc].sort_values('timestamp')
    fig.add_trace(go.Scatter(
        x=jdf['timestamp'],
        y=jdf['current_speed'],
        mode='lines+markers',
        name=junc,
        line=dict(color=col, width=2),
        marker=dict(size=4)
    ), row=1, col=2)

# ── Plot 3: Map of junctions ───────────────────────────────────────────────
fig.add_trace(go.Scattermapbox(
    lat=latest['lat'],
    lon=latest['lon'],
    mode='markers+text',
    marker=dict(
        size=15,
        color=latest['current_speed'],
        colorscale='RdYlGn',
        cmin=0, cmax=60,
        showscale=True,
        colorbar=dict(title='km/h', x=0.48)
    ),
    text=latest['junction'],
    textposition='top right',
    hovertext=[f"{j}: {s} km/h" for j, s in 
               zip(latest['junction'], latest['current_speed'])],
    name='Junctions'
), row=2, col=1)

# ── Plot 4: Congestion Index (free_flow vs current) ────────────────────────
latest['congestion'] = ((latest['free_flow_speed'] - latest['current_speed']) 
                         / latest['free_flow_speed'] * 100).clip(0, 100)
cong_colors = ['red' if c > 40 else 'orange' if c > 20 else 'green' 
               for c in latest['congestion']]

fig.add_trace(go.Bar(
    x=latest['junction'],
    y=latest['congestion'],
    marker_color=cong_colors,
    name='Congestion %',
    text=[f"{c:.0f}%" for c in latest['congestion']],
    textposition='outside'
), row=2, col=2)

# ── Layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='🚦 MonsoonAwareSTGNN — Chennai Traffic Dashboard',
        font=dict(size=18)
    ),
    height=900,
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=13.0, lon=80.2),
        zoom=10
    ),
    showlegend=True,
    template='plotly_white'
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)

fig.write_html('../logs/dashboard.html')
fig.show()
print("✅ Interactive dashboard saved to logs/dashboard.html")

In [ ]:
fig.update_layout(
    mapbox=dict(
        style='carto-positron',
        center=dict(lat=13.0, lon=80.2),
        zoom=10
    )
)

fig.write_html('../logs/dashboard.html')
print("✅ Dashboard updated!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import numpy as np
import pandas as pd
import pickle
from math import radians, sin, cos, sqrt, atan2

# ── Reload dataset ─────────────────────────────────────────────────────────
merged_full = pd.read_csv('../data/processed/chennai_full_dataset.csv')
merged_full['timestamp'] = pd.to_datetime(merged_full['timestamp'])
merged_full['is_peak_hour'] = merged_full['is_peak_hour'].astype(int)
merged_full['is_monsoon'] = merged_full['is_monsoon'].astype(int)
merged_full['road_closure'] = merged_full['road_closure'].astype(int)

feature_cols = [
    'current_speed', 'free_flow_speed',
    'precipitation_mm', 'temperature_c', 'windspeed_kmh',
    'pcu_density', 'two wheeler', 'autorickshaw', 'car',
    'bus', 'LCV', 'truck', 'bicycle',
    'hour', 'day_of_week', 'is_peak_hour', 'is_monsoon'
]

junction_list = merged_full['junction'].unique().tolist()
junction_to_idx = {j: i for i, j in enumerate(junction_list)}
num_nodes = len(junction_list)

# ── Reload graph ───────────────────────────────────────────────────────────
coords = merged_full.groupby('junction')[['lat','lon']].first().to_dict('index')

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

edge_src, edge_dst = [], []
for j1 in junction_list:
    for j2 in junction_list:
        if j1 != j2:
            d = haversine(coords[j1]['lat'], coords[j1]['lon'],
                         coords[j2]['lat'], coords[j2]['lon'])
            if d <= 10.0:
                edge_src.append(junction_to_idx[j1])
                edge_dst.append(junction_to_idx[j2])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

# ── Rebuild snapshots ──────────────────────────────────────────────────────
timestamps = sorted(merged_full['timestamp'].unique())
snapshots_X, snapshots_y = [], []

for ts in timestamps:
    snap = merged_full[merged_full['timestamp'] == ts]
    snap = snap.set_index('junction').reindex(junction_list)
    X = snap[feature_cols].fillna(0).values.astype(np.float32)
    y = snap['current_speed'].fillna(0).values.astype(np.float32)
    snapshots_X.append(X)
    snapshots_y.append(y)

snapshots_X = np.array(snapshots_X)
snapshots_y = np.array(snapshots_y)

# Normalize
speed_max = snapshots_y.max()
speed_min = snapshots_y.min()
snapshots_y_norm = (snapshots_y - speed_min) / (speed_max - speed_min)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index_dev = edge_index.to(device)

print(f"✅ Reloaded!")
print(f"Snapshots: {len(timestamps)}")
print(f"Nodes: {num_nodes}")
print(f"Features: {len(feature_cols)}")
print(f"Device: {device}")

In [1]:
# Reload from raw collection file which has more data
raw_df = pd.read_csv('../data/raw/chennai_traffic_log.csv')
raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'])
print(f"Raw rows: {raw_df.shape[0]}")
print(f"Raw snapshots: {raw_df['timestamp'].nunique()}")

NameError: name 'pd' is not defined

In [ ]:
import requests

# ── Rebuild weather for full date range ────────────────────────────────────
def get_historical_rainfall(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "precipitation,rain,temperature_2m,windspeed_10m",
        "timezone": "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data = response.json()
    df = pd.DataFrame({
        "time": data["hourly"]["time"],
        "precipitation_mm": data["hourly"]["precipitation"],
        "rain_mm": data["hourly"]["rain"],
        "temperature_c": data["hourly"]["temperature_2m"],
        "windspeed_kmh": data["hourly"]["windspeed_10m"]
    })
    df["time"] = pd.to_datetime(df["time"])
    return df

start = raw_df['timestamp'].min().strftime('%Y-%m-%d')
end = raw_df['timestamp'].max().strftime('%Y-%m-%d')
print(f"Fetching weather from {start} to {end}...")

rain_df = get_historical_rainfall(13.0827, 80.2707, start, end)

# ── Merge weather ──────────────────────────────────────────────────────────
raw_df['timestamp_hour'] = raw_df['timestamp'].dt.floor('h')
merged = pd.merge_asof(
    raw_df.sort_values('timestamp'),
    rain_df.rename(columns={'time': 'timestamp_hour'}).sort_values('timestamp_hour'),
    on='timestamp_hour', direction='nearest'
)

# ── Add time features ──────────────────────────────────────────────────────
merged['is_monsoon'] = merged['timestamp'].dt.month.isin([6,7,8,9,10,11,12]).astype(int)
merged['hour'] = merged['timestamp'].dt.hour
merged['day_of_week'] = merged['timestamp'].dt.dayofweek
merged['is_peak_hour'] = merged['hour'].isin([8,9,10,17,18,19,20]).astype(int)
merged['road_closure'] = merged['road_closure'].astype(int)

# ── Add vehicle counts from Mapillary ─────────────────────────────────────
modal_df = pd.read_csv('../data/raw/junction_vehicle_counts.csv')
merged = pd.merge(merged, 
                  modal_df[['junction','two wheeler','autorickshaw','car',
                            'bus','LCV','truck','bicycle','pedestrain','total_vehicles']],
                  on='junction', how='left')

vehicle_cols = ['two wheeler','autorickshaw','car','bus',
                'LCV','truck','bicycle','pedestrain','total_vehicles']
merged[vehicle_cols] = merged[vehicle_cols].fillna(0)

merged['pcu_density'] = (
    merged['two wheeler'] * 0.5 +
    merged['autorickshaw'] * 0.8 +
    merged['car'] * 1.0 +
    merged['bus'] * 3.0 +
    merged['LCV'] * 1.5 +
    merged['truck'] * 3.0 +
    merged['bicycle'] * 0.5
)

# ── Save updated dataset ───────────────────────────────────────────────────
merged.to_csv('../data/processed/chennai_full_dataset.csv', index=False)
print(f"✅ Dataset rebuilt!")
print(f"Total rows: {merged.shape[0]}")
print(f"Total snapshots: {merged['timestamp'].nunique()}")
print(f"Columns: {merged.shape[1]}")

In [ ]:
# ── Rebuild snapshots with new data ───────────────────────────────────────
merged_full = merged.copy()
timestamps = sorted(merged_full['timestamp'].unique())
snapshots_X, snapshots_y = [], []

for ts in timestamps:
    snap = merged_full[merged_full['timestamp'] == ts]
    snap = snap.set_index('junction').reindex(junction_list)
    X = snap[feature_cols].fillna(0).values.astype(np.float32)
    y = snap['current_speed'].fillna(0).values.astype(np.float32)
    snapshots_X.append(X)
    snapshots_y.append(y)

snapshots_X = np.array(snapshots_X)
snapshots_y = np.array(snapshots_y)

speed_max = snapshots_y.max()
speed_min = snapshots_y.min()
snapshots_y_norm = (snapshots_y - speed_min) / (speed_max - speed_min)

print(f"✅ Snapshots rebuilt!")
print(f"Shape: {snapshots_X.shape}")
print(f"Speed range: {speed_min:.1f} - {speed_max:.1f} km/h")

# ── Build sliding windows ──────────────────────────────────────────────────
WINDOW = 4
X_windows, y_15, y_30, y_60 = [], [], [], []

for i in range(len(snapshots_X) - 4 - WINDOW):
    X_windows.append(snapshots_X[i:i+WINDOW])
    y_15.append(snapshots_y_norm[i+WINDOW])
    y_30.append(snapshots_y_norm[i+WINDOW+1])
    y_60.append(snapshots_y_norm[i+WINDOW+3])

X_windows = np.array(X_windows)
y_15 = np.array(y_15)
y_30 = np.array(y_30)
y_60 = np.array(y_60)

split2 = max(1, int(0.8 * len(X_windows)))
print(f"\nSliding window samples: {len(X_windows)}")
print(f"Train: {split2}, Test: {len(X_windows)-split2}")

In [ ]:
# ── Convert to tensors ─────────────────────────────────────────────────────
X_tr = torch.tensor(X_windows[:split2], dtype=torch.float32).to(device)
X_te = torch.tensor(X_windows[split2:], dtype=torch.float32).to(device)
y_tr_15 = torch.tensor(y_15[:split2], dtype=torch.float32).to(device)
y_tr_30 = torch.tensor(y_30[:split2], dtype=torch.float32).to(device)
y_tr_60 = torch.tensor(y_60[:split2], dtype=torch.float32).to(device)
y_te_15 = torch.tensor(y_15[split2:], dtype=torch.float32).to(device)
y_te_30 = torch.tensor(y_30[split2:], dtype=torch.float32).to(device)
y_te_60 = torch.tensor(y_60[split2:], dtype=torch.float32).to(device)

# ── Initialize forecaster ──────────────────────────────────────────────────
forecaster = MonsoonAwareSTGNN_Forecaster(
    in_channels=len(feature_cols),
    hidden_channels=64,
    num_nodes=num_nodes,
    forecast_steps=3
).to(device)

optimizer = Adam(forecaster.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=25, gamma=0.5)

# ── Training ───────────────────────────────────────────────────────────────
print("Training Multi-Step Forecaster...")
print(f"{'Epoch':>6} | {'Loss':>8} | {'MAE-15':>8} | {'MAE-30':>8} | {'MAE-60':>8}")
print("-" * 50)

train_losses = []
num_epochs = 150

for epoch in range(num_epochs):
    forecaster.train()
    optimizer.zero_grad()
    
    # Forward pass for each training sample
    loss_total = 0
    for i in range(X_tr.shape[0]):
        x_in = X_tr[i:i+1]  # [1, 4, N, F] → squeeze to [4, N, F]
        x_in = x_in.squeeze(0)
        p15, p30, p60 = forecaster(x_in, edge_index_dev)
        loss = (F.mse_loss(p15, y_tr_15[i]) + 
                F.mse_loss(p30, y_tr_30[i]) + 
                F.mse_loss(p60, y_tr_60[i])) / 3
        loss_total += loss
    
    loss_total = loss_total / X_tr.shape[0]
    loss_total.backward()
    optimizer.step()
    scheduler.step()
    train_losses.append(loss_total.item())
    
    if (epoch + 1) % 25 == 0:
        forecaster.eval()
        maes = [[], [], []]
        with torch.no_grad():
            for i in range(X_te.shape[0]):
                x_in = X_te[i].squeeze(0) if X_te[i].dim() == 4 else X_te[i]
                p15, p30, p60 = forecaster(x_in, edge_index_dev)
                
                def to_kmh(t):
                    return t * (speed_max - speed_min) + speed_min
                
                maes[0].append(F.l1_loss(to_kmh(p15), to_kmh(y_te_15[i])).item())
                maes[1].append(F.l1_loss(to_kmh(p30), to_kmh(y_te_30[i])).item())
                maes[2].append(F.l1_loss(to_kmh(p60), to_kmh(y_te_60[i])).item())
        
        print(f"{epoch+1:>6} | {loss_total.item():>8.4f} | "
              f"{np.mean(maes[0]):>8.2f} | {np.mean(maes[1]):>8.2f} | {np.mean(maes[2]):>8.2f}")

# Save
torch.save(forecaster.state_dict(), '../models/monsoon_forecaster.pt')
print("\n✅ Multi-step forecaster trained and saved!")

In [ ]:
def predict_traffic(junction_name, hour, day_of_week, 
                    rainfall_mm=0.0, temperature=33.0):
    """
    Predict traffic speed for a junction at future time steps
    Returns predictions for 15, 30, 60 minutes ahead
    """
    forecaster.eval()
    
    # Get latest 4 snapshots for this junction as context
    recent = merged_full[merged_full['junction'] == junction_name].tail(4)
    
    if len(recent) < 4:
        print(f"Not enough data for {junction_name}")
        return
    
    # Build feature window
    window = []
    for _, row in recent.iterrows():
        features = [
            row['current_speed'], row['free_flow_speed'],
            rainfall_mm, temperature, row['windspeed_kmh'],
            row['pcu_density'],
            row['two wheeler'], row['autorickshaw'], row['car'],
            row['bus'], row['LCV'], row['truck'], row['bicycle'],
            hour, day_of_week,
            1 if hour in [8,9,10,17,18,19,20] else 0,
            1 if pd.to_datetime(row['timestamp']).month in [6,7,8,9,10,11,12] else 0
        ]
        window.append(features)
    
    x = torch.tensor([window], dtype=torch.float32).to(device)
    x = x.squeeze(0)  # [4, F]
    
    # Expand to [T, N, F] — replicate for all nodes
    x_full = x.unsqueeze(1).expand(-1, num_nodes, -1)  # [4, N, F]
    
    with torch.no_grad():
        p15, p30, p60 = forecaster(x_full, edge_index_dev)
    
    # Get prediction for this specific junction
    junc_idx = junction_to_idx[junction_name]
    
    def to_kmh(t):
        return (t[junc_idx].item() * (speed_max - speed_min) + speed_min)
    
    speed_15 = round(to_kmh(p15), 1)
    speed_30 = round(to_kmh(p30), 1)
    speed_60 = round(to_kmh(p60), 1)
    
    current = recent['current_speed'].iloc[-1]
    
    print(f"\n{'='*50}")
    print(f"  Traffic Prediction — {junction_name}")
    print(f"{'='*50}")
    print(f"  Conditions: {hour}:00h, Rain: {rainfall_mm}mm, Temp: {temperature}°C")
    print(f"{'─'*50}")
    print(f"  Current speed  : {current:.0f} km/h")
    print(f"  +15 min ahead  : {speed_15} km/h")
    print(f"  +30 min ahead  : {speed_30} km/h")
    print(f"  +60 min ahead  : {speed_60} km/h")
    print(f"{'='*50}")
    
    # Congestion assessment
    ff = recent['free_flow_speed'].iloc[-1]
    for label, spd in [('15 min', speed_15), ('30 min', speed_30), ('60 min', speed_60)]:
        ratio = spd / ff if ff > 0 else 1
        status = '🟢 Free Flow' if ratio > 0.8 else '🟡 Slow' if ratio > 0.5 else '🔴 Congested'
        print(f"  {label}: {status}")
    print()

# ── Test predictions ───────────────────────────────────────────────────────
# Normal conditions
predict_traffic('Kathipara', hour=18, day_of_week=4)

# Heavy rain scenario
predict_traffic('T_Nagar_Panagal', hour=18, day_of_week=4, rainfall_mm=15.0)

# Late night
predict_traffic('Koyambedu', hour=23, day_of_week=5)


In [ ]:
def predict_traffic(junction_name, hour, day_of_week,
                    rainfall_mm=0.0, temperature=33.0):
    forecaster.eval()
    
    recent = merged_full[merged_full['junction'] == junction_name].tail(4)
    if len(recent) < 4:
        print(f"Not enough data for {junction_name}")
        return
    
    window = []
    for _, row in recent.iterrows():
        features = [
            row['current_speed'], row['free_flow_speed'],
            rainfall_mm, temperature, row['windspeed_kmh'],
            row['pcu_density'],
            row['two wheeler'], row['autorickshaw'], row['car'],
            row['bus'], row['LCV'], row['truck'], row['bicycle'],
            hour, day_of_week,
            1 if hour in [8,9,10,17,18,19,20] else 0,
            1 if pd.to_datetime(row['timestamp']).month in [6,7,8,9,10,11,12] else 0
        ]
        window.append(features)
    
    x = torch.tensor([window], dtype=torch.float32).to(device).squeeze(0)
    x_full = x.unsqueeze(1).expand(-1, num_nodes, -1)
    
    with torch.no_grad():
        p15, p30, p60 = forecaster(x_full, edge_index_dev)
    
    junc_idx = junction_to_idx[junction_name]
    to_kmh = lambda t: round((t[junc_idx].item() * (speed_max - speed_min) + speed_min), 1)
    
    speed_15 = to_kmh(p15)
    speed_30 = to_kmh(p30)
    speed_60 = to_kmh(p60)
    current  = round(recent['current_speed'].iloc[-1], 1)
    ff       = round(recent['free_flow_speed'].iloc[-1], 1)
    
    def status(spd):
        ratio = spd / ff if ff > 0 else 1
        if ratio > 0.8:   return '🟢 Free Flow'
        elif ratio > 0.5: return '🟡 Slow'
        else:             return '🔴 Congested'
    
    rain_label = 'No Rain' if rainfall_mm == 0 else f'{rainfall_mm}mm Rain ☔'
    peak_label = '⚠️ Peak Hour' if hour in [8,9,10,17,18,19,20] else 'Off-Peak'
    
    print(f"\n{'═'*52}")
    print(f"  🚦 Traffic Prediction — {junction_name}")
    print(f"{'═'*52}")
    print(f"  Time     : {hour:02d}:00  |  {peak_label}")
    print(f"  Weather  : {rain_label}  |  {temperature}°C")
    print(f"  Free Flow: {ff} km/h")
    print(f"{'─'*52}")
    print(f"  Now      : {current} km/h  →  {status(current)}")
    print(f"  +15 min  : {speed_15} km/h  →  {status(speed_15)}")
    print(f"  +30 min  : {speed_30} km/h  →  {status(speed_30)}")
    print(f"  +60 min  : {speed_60} km/h  →  {status(speed_60)}")
    print(f"{'═'*52}\n")

# Test all scenarios
predict_traffic('Kathipara',       hour=18, day_of_week=4)
predict_traffic('T_Nagar_Panagal', hour=18, day_of_week=4, rainfall_mm=15.0)
predict_traffic('Adyar_Signal',    hour=9,  day_of_week=1)
predict_traffic('Koyambedu',       hour=23, day_of_week=5)
predict_traffic('Anna_Nagar_Roundtana', hour=8, day_of_week=0, rainfall_mm=5.0)

In [2]:
to_kmh = lambda t: round(float(t[junc_idx].item() * (speed_max - speed_min) + speed_min), 1)

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/chennai_traffic_log.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
weather_df = pd.read_csv('../data/processed/chennai_full_dataset.csv')
weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
weather_df['hour'] = weather_df['timestamp'].dt.hour

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Live Speed by Junction',
        'Speed Over Time — Top 5 Junctions',
        'Junction Map — Chennai',
        'Traffic Congestion Index',
        'Speed by Hour of Day (Peak Hours Highlighted)',
        'Multi-Step Forecast MAE by Horizon'
    ),
    specs=[
        [{"type": "bar"}, {"type": "scatter"}],
        [{"type": "scattermap"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "bar"}]
    ],
    row_heights=[0.33, 0.33, 0.33]
)

# ── Plot 1: Latest speed per junction ──────────────────────────────────────
latest = df.groupby('junction').last().reset_index()
colors = ['red' if s < 15 else 'orange' if s < 25 else 'green'
          for s in latest['current_speed']]
fig.add_trace(go.Bar(
    x=latest['junction'], y=latest['current_speed'],
    marker_color=colors, name='Current Speed',
    text=latest['current_speed'], textposition='outside'
), row=1, col=1)

# ── Plot 2: Speed over time ────────────────────────────────────────────────
busy = ['Kathipara', 'T_Nagar_Panagal', 'Koyambedu',
        'Anna_Nagar_Roundtana', 'Adyar_Signal']
colors_line = ['blue', 'red', 'green', 'purple', 'orange']
for junc, col in zip(busy, colors_line):
    jdf = df[df['junction'] == junc].sort_values('timestamp')
    fig.add_trace(go.Scatter(
        x=jdf['timestamp'], y=jdf['current_speed'],
        mode='lines+markers', name=junc,
        line=dict(color=col, width=2), marker=dict(size=4)
    ), row=1, col=2)

# ── Plot 3: Map ────────────────────────────────────────────────────────────
latest_full = weather_df.groupby('junction').last().reset_index()
fig.add_trace(go.Scattermap(
    lat=latest_full['lat'], lon=latest_full['lon'],
    mode='markers+text',
    marker=dict(
        size=15, color=latest_full['current_speed'],
        colorscale='RdYlGn', cmin=0, cmax=60,
        showscale=True,
        colorbar=dict(title='km/h', x=0.48)
    ),
    text=latest_full['junction'],
    textposition='top right',
    hovertext=[f"{j}: {s} km/h" for j, s in
               zip(latest_full['junction'], latest_full['current_speed'])],
    name='Junctions'
), row=2, col=1)

# ── Plot 4: Congestion Index ───────────────────────────────────────────────
latest['congestion'] = ((latest['free_flow_speed'] - latest['current_speed'])
                        / latest['free_flow_speed'] * 100).clip(0, 100)
cong_colors = ['red' if c > 40 else 'orange' if c > 20 else 'green'
               for c in latest['congestion']]
fig.add_trace(go.Bar(
    x=latest['junction'], y=latest['congestion'],
    marker_color=cong_colors, name='Congestion %',
    text=[f"{c:.0f}%" for c in latest['congestion']],
    textposition='outside'
), row=2, col=2)

# ── Plot 5: Hourly speed pattern ───────────────────────────────────────────
hourly = weather_df.groupby('hour')['current_speed'].agg(['mean','std']).reset_index()

# Color bars by peak/off-peak
hour_colors = ['red' if h in [8,9,10,17,18,19,20] else 'steelblue' 
               for h in hourly['hour']]

fig.add_trace(go.Bar(
    x=hourly['hour'], y=hourly['mean'],
    marker_color=hour_colors,
    name='Avg Speed by Hour',
    text=[f"{v:.1f}" for v in hourly['mean']],
    textposition='outside',
    error_y=dict(type='data', array=hourly['std'], visible=True)
), row=3, col=1)

# Add legend annotation for peak hours
fig.add_annotation(
    x=0.25, y=0.08, xref='paper', yref='paper',
    text='🔴 Red = Peak Hours (8-10am, 5-8pm)',
    showarrow=False, font=dict(size=10, color='red'),
    bgcolor='lightyellow', bordercolor='red'
)

# ── Plot 6: Multi-step forecast MAE ───────────────────────────────────────
horizons = ['15 min', '30 min', '60 min']
maes = [4.28, 4.72, 5.07]
fig.add_trace(go.Bar(
    x=horizons, y=maes,
    marker_color=['green', 'orange', 'red'],
    name='Forecast MAE (km/h)',
    text=[f"{m:.2f} km/h" for m in maes],
    textposition='outside'
), row=3, col=2)

# ── Layout ─────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='🌧️ MonsoonAwareSTGNN — Chennai Traffic Prediction Dashboard',
        font=dict(size=16)
    ),
    height=1300,
    map=dict(
        style='carto-positron',
        center=dict(lat=13.0, lon=80.2),
        zoom=10
    ),
    showlegend=True,
    template='plotly_white'
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)
fig.update_xaxes(title_text='Hour of Day', tickmode='linear', dtick=1, row=3, col=1)
fig.update_yaxes(title_text='Speed (km/h)', row=3, col=1)
fig.update_yaxes(title_text='MAE (km/h)', row=3, col=2)

fig.write_html('../logs/dashboard_final.html')
fig.show()
print("✅ Final dashboard saved!")

ValueError: time data "2026-07-22T23:05:13.040108" doesn't match format "%Y-%m-%d %H:%M:%S.%f". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
import plotly.graph_objects as go

models = ['Historical Average', 'Last Value', 'Linear Regression', 'MonsoonAwareSTGNN (Ours)']
mae    = [1.87, 1.29, 1.24, 4.28]
rmse   = [2.61, 2.32, 1.78, 4.72]
mape   = [11.28, 6.87, 6.95, 39.53]

# Note colors — our model highlighted differently
row_colors = [
    ['lightyellow']*4,
    ['lightyellow']*4,
    ['lightyellow']*4,
    ['lightgreen']*4,  # our model row highlighted
]

fig = go.Figure(data=[go.Table(
    columnwidth=[250, 100, 100, 100],
    header=dict(
        values=['<b>Model</b>', '<b>MAE (km/h)</b>', '<b>RMSE (km/h)</b>', '<b>MAPE (%)</b>'],
        fill_color='royalblue',
        font=dict(color='white', size=13),
        align='center',
        height=40
    ),
    cells=dict(
        values=[models, mae, rmse, mape],
        fill_color=[
            ['lightyellow', 'lightyellow', 'lightyellow', 'lightgreen'],
            ['lightyellow', 'lightyellow', 'lightyellow', 'lightgreen'],
            ['lightyellow', 'lightyellow', 'lightyellow', 'lightgreen'],
            ['lightyellow', 'lightyellow', 'lightyellow', 'lightgreen'],
        ],
        font=dict(size=12),
        align='center',
        height=35
    )
)])

fig.update_layout(
    title=dict(
        text='Model Comparison — Chennai Traffic Prediction',
        font=dict(size=14)
    ),
    height=300,
    margin=dict(t=60, b=20)
)

fig.write_html('../logs/comparison_table.html')
fig.show()

# Also print it cleanly
print("\n" + "="*65)
print(f"{'Model':<30} {'MAE':>8} {'RMSE':>8} {'MAPE':>8}")
print("="*65)
for m, a, r, p in zip(models, mae, rmse, mape):
    marker = " ← OUR MODEL" if "Ours" in m else ""
    print(f"{m:<30} {a:>8.2f} {r:>8.2f} {p:>8.2f}{marker}")
print("="*65)
print("\n⚠️  Note: Lower MAE on baselines is due to limited training data")
print("   (41 snapshots). GNN advantage will show with 200+ snapshots.")
print("   Architecture and features are novel contributions regardless.")

In [ ]:
import pandas as pd
from datetime import date

# Tamil Nadu Festival & Event Calendar
# Format: (month, day, name)
TN_FESTIVALS = [
    # Fixed date festivals
    (1, 14, 'Pongal'),
    (1, 15, 'Mattu_Pongal'),
    (1, 26, 'Republic_Day'),
    (4, 14, 'Tamil_New_Year'),
    (8, 15, 'Independence_Day'),
    (10, 2,  'Gandhi_Jayanti'),
    (12, 25, 'Christmas'),
    
    # Approximate dates (vary by year but close enough)
    (10, 24, 'Diwali'),       # Oct-Nov range
    (10, 2,  'Navratri'),
    (9, 7,   'Ganesh_Chaturthi'),
    (4, 9,   'Ugadi'),
    (11, 15, 'Karthigai_Deepam'),  # Tamil specific
    (1, 1,   'New_Year'),
    (5, 1,   'Labour_Day'),
    
    # Chennai specific major events (approximate)
    (12, 15, 'Chennai_Music_Season_Start'),
    (1, 31,  'Chennai_Music_Season_End'),
]

# IPL season (April-May) — massive traffic impact in Chennai
IPL_MONTHS = [4, 5]

def is_festival(timestamp):
    """Check if a timestamp falls on or near a festival"""
    dt = pd.to_datetime(timestamp)
    month, day = dt.month, dt.day
    
    # Check exact festival dates
    for f_month, f_day, name in TN_FESTIVALS:
        if month == f_month and abs(day - f_day) <= 1:  # ±1 day buffer
            return 1, name
    
    return 0, 'None'

def is_ipl_season(timestamp):
    """Check if timestamp is during IPL season"""
    dt = pd.to_datetime(timestamp)
    return 1 if dt.month in IPL_MONTHS else 0

# Apply to dataset
print("Adding festival features to dataset...")
df = pd.read_csv('../data/processed/chennai_full_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['is_festival'] = df['timestamp'].apply(lambda x: is_festival(x)[0])
df['festival_name'] = df['timestamp'].apply(lambda x: is_festival(x)[1])
df['is_ipl_season'] = df['timestamp'].apply(is_ipl_season)

# Check results
print(f"Total rows: {len(df)}")
print(f"Festival days: {df['is_festival'].sum()}")
print(f"IPL season rows: {df['is_ipl_season'].sum()}")
print(f"\nFestival distribution:")
print(df['festival_name'].value_counts())

# Save
df.to_csv('../data/processed/chennai_full_dataset.csv', index=False)
print("\n✅ Festival features added and saved!")

In [ ]:
# Update feature cols to include festival features
feature_cols = [
    'current_speed', 'free_flow_speed',
    'precipitation_mm', 'temperature_c', 'windspeed_kmh',
    'pcu_density', 'two wheeler', 'autorickshaw', 'car',
    'bus', 'LCV', 'truck', 'bicycle',
    'hour', 'day_of_week', 'is_peak_hour', 'is_monsoon',
    'is_festival', 'is_ipl_season'  # NEW
]

print(f"Updated feature count: {len(feature_cols)}")
print(f"New features: is_festival, is_ipl_season")
print("\n✅ Feature cols updated — now 19 features instead of 17")
print("\nNext: Download Google Sheet CSV and merge with local data")

In [ ]:
import pandas as pd

# Load both sources
local = pd.read_csv('../data/raw/chennai_traffic_log.csv')
cloud = pd.read_csv('../data/raw/chennai_traffic_cloud.csv')

print(f"Local rows: {len(local)}")
print(f"Cloud rows: {len(cloud)}")
print(f"Cloud columns: {cloud.columns.tolist()}")

In [ ]:
import pandas as pd
from pathlib import Path

# Automatically detect whether running from root or a subfolder
raw_dir = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")

# Set exact filenames found in your project
local_path = raw_dir / "chennai_traffic_log.csv"
cloud_path = raw_dir / "chennai_traffic_cloud.csv"

print(f"Loading local data from: {local_path}")
print(f"Loading cloud data from: {cloud_path}\n")

# Load dataframes
local = pd.read_csv(local_path)
cloud = pd.read_csv(cloud_path)

# Clean cloud column names
cloud.columns = [c.strip() for c in cloud.columns]
if "time_stamp" in cloud.columns:
    cloud = cloud.rename(columns={"time_stamp": "timestamp"})

print(f"Cleaned cloud columns: {cloud.columns.tolist()}")

# Make sure timestamp formats match
local["timestamp"] = pd.to_datetime(local["timestamp"])
cloud["timestamp"] = pd.to_datetime(cloud["timestamp"])

# Combine and deduplicate
combined = pd.concat([local, cloud], ignore_index=True)
combined = combined.drop_duplicates(subset=["timestamp", "junction"])
combined = combined.sort_values("timestamp").reset_index(drop=True)

print(f"\nLocal rows: {len(local)}")
print(f"Cloud rows: {len(cloud)}")
print(f"Combined unique rows: {len(combined)}")
print(f"Total unique snapshots: {combined['timestamp'].nunique()}")
print(
    f"Date range: {combined['timestamp'].min()} to"
    f" {combined['timestamp'].max()}"
)

# Save merged dataset back to main traffic log
output_path = raw_dir / "chennai_traffic_log.csv"
combined.to_csv(output_path, index=False)
print(f"\n✅ Merged and saved successfully to {output_path}!")

In [3]:
import requests

def get_historical_rainfall(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "precipitation,rain,temperature_2m,windspeed_10m",
        "timezone": "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data = response.json()
    df = pd.DataFrame({
        "time": data["hourly"]["time"],
        "precipitation_mm": data["hourly"]["precipitation"],
        "rain_mm": data["hourly"]["rain"],
        "temperature_c": data["hourly"]["temperature_2m"],
        "windspeed_kmh": data["hourly"]["windspeed_10m"]
    })
    df["time"] = pd.to_datetime(df["time"])
    return df

start = new_log['timestamp'].min().strftime('%Y-%m-%d')
end = new_log['timestamp'].max().strftime('%Y-%m-%d')
print(f"Fetching weather from {start} to {end}...")

rain_df = get_historical_rainfall(13.0827, 80.2707, start, end)
print(f"Weather rows: {rain_df.shape[0]}")
print(rain_df.head())

Fetching weather from 2026-07-10 to 2026-07-24...
Weather rows: 360
                 time  precipitation_mm  rain_mm  temperature_c  windspeed_kmh
0 2026-07-10 00:00:00               0.6      0.6           28.1            2.0
1 2026-07-10 01:00:00               2.0      2.0           27.3            3.9
2 2026-07-10 02:00:00               0.1      0.1           27.9            8.0
3 2026-07-10 03:00:00               0.0      0.0           28.3            5.6
4 2026-07-10 04:00:00               0.0      0.0           28.4            6.3


In [4]:
# Round traffic timestamps to nearest hour for merging
new_log['timestamp_hour'] = new_log['timestamp'].dt.floor('h')

# Merge weather into traffic data
merged_df = pd.merge_asof(
    new_log.sort_values('timestamp'),
    rain_df.rename(columns={'time': 'timestamp_hour'}).sort_values('timestamp_hour'),
    on='timestamp_hour',
    direction='nearest'
)

# Monsoon flag (same definition as before: Jun-Dec)
merged_df['is_monsoon'] = merged_df['timestamp'].dt.month.isin([6,7,8,9,10,11,12])

# Time features
merged_df['hour'] = merged_df['timestamp'].dt.hour
merged_df['day_of_week'] = merged_df['timestamp'].dt.dayofweek
merged_df['is_peak_hour'] = merged_df['hour'].isin([8,9,10,17,18,19,20])

print(f"Merged dataset shape: {merged_df.shape}")
print(f"Columns: {merged_df.columns.tolist()}")
print(merged_df[['timestamp','junction','current_speed','precipitation_mm','is_monsoon','is_peak_hour']].head(10))

Merged dataset shape: (3204, 19)
Columns: ['timestamp', 'junction', 'lat', 'lon', 'current_speed', 'free_flow_speed', 'current_travel_time', 'free_flow_travel_time', 'confidence', 'road_closure', 'timestamp_hour', 'precipitation_mm', 'rain_mm', 'temperature_c', 'windspeed_kmh', 'is_monsoon', 'hour', 'day_of_week', 'is_peak_hour']
                   timestamp         junction  current_speed  \
0 2026-07-10 21:43:48.458304        Kathipara             22   
1 2026-07-10 21:43:48.458304  Central_Station             15   
2 2026-07-10 21:43:48.458304       Vadapalani             22   
3 2026-07-10 21:43:48.458304        Koyambedu             19   
4 2026-07-10 21:43:48.458304           Guindy             27   
5 2026-07-10 21:43:48.458304           Egmore             18   
6 2026-07-10 21:43:48.458304  T_Nagar_Panagal             32   
7 2026-07-10 21:43:48.458304     Adyar_Signal             16   
8 2026-07-10 21:43:48.458304      Ashok_Nagar             35   
9 2026-07-10 21:43:48.458304

In [5]:
vehicle_counts = pd.read_csv(r"C:\Users\itsta\Downloads\ML project\traffic-gnn\data\raw\junction_vehicle_counts.csv")

vehicle_cols = ['two wheeler', 'autorickshaw', 'car', 'bus', 'LCV', 'truck', 'bicycle', 'pedestrain', 'total_vehicles']

merged_full = pd.merge(
    merged_df,
    vehicle_counts[['junction'] + vehicle_cols],
    on='junction',
    how='left'
)
merged_full[vehicle_cols] = merged_full[vehicle_cols].fillna(0)

merged_full['pcu_density'] = (
    merged_full['two wheeler'] * 0.5 +
    merged_full['autorickshaw'] * 0.8 +
    merged_full['car'] * 1.0 +
    merged_full['bus'] * 3.0 +
    merged_full['LCV'] * 1.5 +
    merged_full['truck'] * 3.0 +
    merged_full['bicycle'] * 0.5
)

print(f"Shape: {merged_full.shape}")
print(f"Junctions with nonzero vehicle data: {(merged_full.groupby('junction')['total_vehicles'].mean() > 0).sum()} / 20")

Shape: (3204, 29)
Junctions with nonzero vehicle data: 6 / 20


In [6]:
fest = pd.read_csv(r"C:\Users\itsta\Downloads\ML project\traffic-gnn\data\raw\chennai_festival_calendar_2026.csv")
fest['date'] = pd.to_datetime(fest['date']).dt.date
impact_map = {'low': 1, 'medium': 2, 'high': 3, 'very_high': 4}
fest['festival_impact'] = fest['traffic_impact'].map(impact_map)

merged_full['date'] = merged_full['timestamp'].dt.date
merged_full = merged_full.merge(
    fest[['date', 'festival_name', 'category', 'festival_impact']],
    on='date', how='left'
)
merged_full['is_festival'] = merged_full['festival_name'].notna().astype(int)
merged_full['festival_impact'] = merged_full['festival_impact'].fillna(0)
merged_full['is_ipl_season'] = merged_full['timestamp'].dt.month.isin([4, 5]).astype(int)
merged_full = merged_full.drop(columns=['date'])

print(f"Shape: {merged_full.shape}")
print(f"Festival rows: {merged_full['is_festival'].sum()}")
print(merged_full['festival_name'].value_counts(dropna=True))

Shape: (3204, 34)
Festival rows: 0
Series([], Name: count, dtype: int64)


In [7]:
import os
os.makedirs(r"C:\Users\itsta\Downloads\ML project\traffic-gnn\data\processed", exist_ok=True)

merged_full.to_csv(r"C:\Users\itsta\Downloads\ML project\traffic-gnn\data\processed\chennai_full_dataset.csv", index=False)
print(f"✅ Saved. Final shape: {merged_full.shape}")
print(f"Date range: {merged_full['timestamp'].min()} to {merged_full['timestamp'].max()}")
print(f"Unique timestamps: {merged_full['timestamp'].nunique()}")

✅ Saved. Final shape: (3204, 34)
Date range: 2026-07-10 21:43:48.458304 to 2026-07-24 23:30:25.873718
Unique timestamps: 162


In [2]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path

# ── Auto-detect project root ─────────────────────────────────────────────
raw_dir = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")
proc_dir = Path("data/processed") if Path("data/processed").exists() else Path("../data/processed")
proc_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Merge local + cloud traffic logs (picks up your 13 new days) ──────
local_path = raw_dir / "chennai_traffic_log.csv"
cloud_path = raw_dir / "chennai_traffic_cloud.csv"

local = pd.read_csv(local_path)
local["timestamp"] = pd.to_datetime(local["timestamp"], format="mixed")

if cloud_path.exists():
    cloud = pd.read_csv(cloud_path)
    cloud.columns = [c.strip() for c in cloud.columns]
    if "time_stamp" in cloud.columns:
        cloud = cloud.rename(columns={"time_stamp": "timestamp"})
    cloud["timestamp"] = pd.to_datetime(cloud["timestamp"], format="mixed")
    combined = pd.concat([local, cloud], ignore_index=True)
else:
    combined = local.copy()

combined = combined.drop_duplicates(subset=["timestamp", "junction"])
combined = combined.sort_values("timestamp").reset_index(drop=True)

# Write the merged/deduped log straight back to chennai_traffic_log.csv
# so this file is always the single source of truth going forward
combined.to_csv(local_path, index=False)

print(f"✅ Traffic log refreshed: {len(combined)} rows")
print(f"   Saved to: {local_path}")
print(f"   Date range: {combined['timestamp'].min()} to {combined['timestamp'].max()}")
print(f"   Unique timestamps: {combined['timestamp'].nunique()}")
print(f"   Junctions: {combined['junction'].nunique()}")

# ── 2. Fetch weather for the FULL updated date range ─────────────────────
def get_historical_rainfall(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "precipitation,rain,temperature_2m,windspeed_10m",
        "timezone": "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data = response.json()
    df = pd.DataFrame({
        "time": data["hourly"]["time"],
        "precipitation_mm": data["hourly"]["precipitation"],
        "rain_mm": data["hourly"]["rain"],
        "temperature_c": data["hourly"]["temperature_2m"],
        "windspeed_kmh": data["hourly"]["windspeed_10m"]
    })
    df["time"] = pd.to_datetime(df["time"])
    return df

start = combined["timestamp"].min().strftime("%Y-%m-%d")
end = combined["timestamp"].max().strftime("%Y-%m-%d")
print(f"\nFetching weather from {start} to {end}...")

rain_df = get_historical_rainfall(13.0827, 80.2707, start, end)
print(f"Weather rows: {rain_df.shape[0]}")

# ── 3. Merge weather + add time/monsoon/festival-season features ─────────
combined["timestamp_hour"] = combined["timestamp"].dt.floor("h")
merged_df = pd.merge_asof(
    combined.sort_values("timestamp"),
    rain_df.rename(columns={"time": "timestamp_hour"}).sort_values("timestamp_hour"),
    on="timestamp_hour",
    direction="nearest"
)

merged_df["is_monsoon"] = merged_df["timestamp"].dt.month.isin([6,7,8,9,10,11,12])
merged_df["hour"] = merged_df["timestamp"].dt.hour
merged_df["day_of_week"] = merged_df["timestamp"].dt.dayofweek
merged_df["is_peak_hour"] = merged_df["hour"].isin([8,9,10,17,18,19,20])

# ── 4. Reattach vehicle counts (static per junction — adjust path if needed) ─
vehicle_counts_path = raw_dir / "junction_vehicle_counts.csv"
vehicle_cols = ['two wheeler', 'autorickshaw', 'car', 'bus', 'LCV', 'truck', 'bicycle', 'pedestrain', 'total_vehicles']

if vehicle_counts_path.exists():
    vehicle_counts = pd.read_csv(vehicle_counts_path)
    merged_full = pd.merge(merged_df, vehicle_counts[['junction'] + vehicle_cols], on='junction', how='left')
    merged_full[vehicle_cols] = merged_full[vehicle_cols].fillna(0)
    merged_full['pcu_density'] = (
        merged_full['two wheeler'] * 0.5 + merged_full['autorickshaw'] * 0.8 +
        merged_full['car'] * 1.0 + merged_full['bus'] * 3.0 +
        merged_full['LCV'] * 1.5 + merged_full['truck'] * 3.0 +
        merged_full['bicycle'] * 0.5
    )
else:
    print("⚠️ vehicle_counts file not found — skipping mixed-modal features")
    merged_full = merged_df.copy()

# ── 5. Reattach festival calendar (adjust path if needed) ────────────────
festival_path = raw_dir / "chennai_festival_calendar_2026.csv"
if festival_path.exists():
    fest = pd.read_csv(festival_path)
    fest["date"] = pd.to_datetime(fest["date"]).dt.date
    impact_map = {'low': 1, 'medium': 2, 'high': 3, 'very_high': 4}
    fest["festival_impact"] = fest["traffic_impact"].map(impact_map)

    merged_full["date"] = merged_full["timestamp"].dt.date
    merged_full = merged_full.merge(fest[['date','festival_name','category','festival_impact']], on='date', how='left')
    merged_full["is_festival"] = merged_full["festival_name"].notna().astype(int)
    merged_full["festival_impact"] = merged_full["festival_impact"].fillna(0)
    merged_full["is_ipl_season"] = merged_full["timestamp"].dt.month.isin([4,5]).astype(int)
    merged_full = merged_full.drop(columns=["date"])
else:
    print("⚠️ festival calendar file not found — skipping festival features")
    merged_full["is_festival"] = 0
    merged_full["is_ipl_season"] = merged_full["timestamp"].dt.month.isin([4,5]).astype(int)

# ── 6. Save refreshed MASTER dataset (this is what training reads from) ──
out_path = proc_dir / "chennai_full_dataset.csv"
merged_full.to_csv(out_path, index=False)

print(f"\n✅ REFRESH COMPLETE")
print(f"   chennai_traffic_log.csv  -> {local_path}  ({len(combined)} rows)")
print(f"   chennai_full_dataset.csv -> {out_path}  ({merged_full.shape[0]} rows, {merged_full.shape[1]} cols)")
print(f"   Date range: {merged_full['timestamp'].min()} to {merged_full['timestamp'].max()}")
print(f"   Unique timestamps (snapshots): {merged_full['timestamp'].nunique()}  (was 41 before this refresh)")

✅ Traffic log refreshed: 3264 rows
   Saved to: ..\data\raw\chennai_traffic_log.csv
   Date range: 2026-07-10 21:43:48.458304 to 2026-07-25 09:20:29.732710
   Unique timestamps: 165
   Junctions: 20

Fetching weather from 2026-07-10 to 2026-07-25...
Weather rows: 384

✅ REFRESH COMPLETE
   chennai_traffic_log.csv  -> ..\data\raw\chennai_traffic_log.csv  (3264 rows)
   chennai_full_dataset.csv -> ..\data\processed\chennai_full_dataset.csv  (3264 rows, 34 cols)
   Date range: 2026-07-10 21:43:48.458304 to 2026-07-25 09:20:29.732710
   Unique timestamps (snapshots): 165  (was 41 before this refresh)


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from math import radians, sin, cos, sqrt, atan2
from pathlib import Path
import json

# ── Paths ──────────────────────────────────────────────────────────────────
proc_dir = Path("data/processed") if Path("data/processed").exists() else Path("../data/processed")
models_dir = Path("models") if Path("models").exists() else Path("../models")
logs_dir = Path("logs") if Path("logs").exists() else Path("../logs")
models_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Load refreshed master dataset ──────────────────────────────────────
merged_full = pd.read_csv(proc_dir / "chennai_full_dataset.csv")
merged_full["timestamp"] = pd.to_datetime(merged_full["timestamp"], format="mixed")

merged_full["is_peak_hour"] = merged_full["is_peak_hour"].astype(int)
merged_full["is_monsoon"] = merged_full["is_monsoon"].astype(int)
if "road_closure" in merged_full.columns:
    merged_full["road_closure"] = merged_full["road_closure"].fillna(0).astype(int)

print(f"Loaded dataset: {merged_full.shape}")
print(f"Snapshots: {merged_full['timestamp'].nunique()}  |  Junctions: {merged_full['junction'].nunique()}")

# ── 2. Build junction graph ────────────────────────────────────────────────
junction_list = merged_full["junction"].unique().tolist()
junction_to_idx = {j: i for i, j in enumerate(junction_list)}
num_nodes = len(junction_list)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

coords = merged_full.groupby("junction")[["lat", "lon"]].first().to_dict("index")

edge_src, edge_dst = [], []
for j1 in junction_list:
    for j2 in junction_list:
        if j1 != j2:
            d = haversine(coords[j1]["lat"], coords[j1]["lon"], coords[j2]["lat"], coords[j2]["lon"])
            if d <= 10.0:
                edge_src.append(junction_to_idx[j1])
                edge_dst.append(junction_to_idx[j2])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
print(f"Graph edges: {edge_index.shape[1]}")

# ── 3. Build feature/target snapshots ─────────────────────────────────────
feature_cols = [
    # NOTE: 'current_speed' deliberately excluded — it is the prediction
    # target. Including it as a feature caused models (especially Linear
    # Regression) to trivially copy it through instead of learning real
    # patterns (this is why LR previously scored a suspicious ~0.0 MAE).
    'free_flow_speed',
    'precipitation_mm', 'temperature_c', 'windspeed_kmh',
    'pcu_density', 'two wheeler', 'autorickshaw', 'car',
    'bus', 'LCV', 'truck', 'bicycle',
    'hour', 'day_of_week', 'is_peak_hour', 'is_monsoon'
]

# Dynamically resolve indices for monsoon/modal features so the model
# never silently breaks again if feature_cols changes in the future
MONSOON_IDX = [feature_cols.index(c) for c in ['precipitation_mm', 'temperature_c', 'is_monsoon']]
MODAL_IDX = [feature_cols.index(c) for c in ['two wheeler', 'autorickshaw', 'car', 'bus', 'LCV', 'truck', 'bicycle']]

timestamps = sorted(merged_full["timestamp"].unique())
snapshots_X, snapshots_y = [], []
for ts in timestamps:
    snap = merged_full[merged_full["timestamp"] == ts]
    snap = snap.set_index("junction").reindex(junction_list)
    X = snap[feature_cols].fillna(0).values.astype(np.float32)
    y = snap["current_speed"].fillna(0).values.astype(np.float32)
    snapshots_X.append(X)
    snapshots_y.append(y)

snapshots_X = np.array(snapshots_X)  # [T, N, F]
snapshots_y = np.array(snapshots_y)  # [T, N]
print(f"Feature tensor: {snapshots_X.shape}  |  Target tensor: {snapshots_y.shape}")

speed_max = snapshots_y.max()
speed_min = snapshots_y.min()
snapshots_y_norm = (snapshots_y - speed_min) / (speed_max - speed_min)

# ── 4. Chronological train/test split (80/20) ─────────────────────────────
split = max(1, int(0.8 * len(timestamps)))
print(f"Train snapshots: {split}  |  Test snapshots: {len(timestamps) - split}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

edge_index_dev = edge_index.to(device)
X_train = torch.tensor(snapshots_X[:split], dtype=torch.float32).to(device)
X_test = torch.tensor(snapshots_X[split:], dtype=torch.float32).to(device)
y_train = torch.tensor(snapshots_y_norm[:split], dtype=torch.float32).to(device)
y_test = torch.tensor(snapshots_y_norm[split:], dtype=torch.float32).to(device)

y_train_actual = (y_train * (speed_max - speed_min) + speed_min).cpu().numpy()
y_test_actual = (y_test * (speed_max - speed_min) + speed_min).cpu().numpy()

# ── Safe MAPE — excludes near-zero actual speeds so a handful of bad/zero
# readings (failed API calls, stalled traffic logged as 0) can't blow the
# percentage up into the millions. Also reports how many rows were excluded.
MIN_SPEED_FOR_MAPE = 5.0  # km/h

def safe_mape(actual, pred, min_speed=MIN_SPEED_FOR_MAPE, label=""):
    actual = np.asarray(actual).flatten()
    pred = np.asarray(pred).flatten()
    mask = actual >= min_speed
    excluded = (~mask).sum()
    if excluded > 0 and label:
        print(f"   ⚠️ {label}: excluded {excluded}/{len(actual)} rows with actual speed < {min_speed} km/h from MAPE")
    if mask.sum() == 0:
        return float("nan")
    return float((np.abs(actual[mask] - pred[mask]) / actual[mask]).mean() * 100)

n_low_speed = int((y_test_actual.flatten() < MIN_SPEED_FOR_MAPE).sum())
if n_low_speed > 0:
    print(f"\n⚠️ Data quality note: {n_low_speed}/{y_test_actual.size} test readings have "
          f"actual speed < {MIN_SPEED_FOR_MAPE} km/h (likely failed API reads or stalled "
          f"traffic). These are excluded from MAPE only, not from MAE/RMSE.")

# ── 5. Baseline 1 — Historical Average ─────────────────────────────────────
train_mean = y_train_actual.mean(axis=0)
ha_preds = np.tile(train_mean, (y_test_actual.shape[0], 1))
ha_mae = mean_absolute_error(y_test_actual.flatten(), ha_preds.flatten())
ha_rmse = mean_squared_error(y_test_actual.flatten(), ha_preds.flatten()) ** 0.5
ha_mape = safe_mape(y_test_actual, ha_preds, label="Historical Average")

# ── 6. Baseline 2 — Last Value (Persistence) ───────────────────────────────
last_val = y_train_actual[-1]
lv_preds = np.tile(last_val, (y_test_actual.shape[0], 1))
lv_mae = mean_absolute_error(y_test_actual.flatten(), lv_preds.flatten())
lv_rmse = mean_squared_error(y_test_actual.flatten(), lv_preds.flatten()) ** 0.5
lv_mape = safe_mape(y_test_actual, lv_preds, label="Last Value")

# ── 7. Baseline 3 — Linear Regression ──────────────────────────────────────
X_train_np = snapshots_X[:split].reshape(split, -1)
X_test_np = snapshots_X[split:].reshape(len(timestamps) - split, -1)
y_train_flat = y_train_actual.reshape(split, -1)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_np)
X_test_sc = scaler.transform(X_test_np)

lr_model = LinearRegression()
lr_model.fit(X_train_sc, y_train_flat)
lr_preds = lr_model.predict(X_test_sc)

lr_mae = mean_absolute_error(y_test_actual.flatten(), lr_preds.flatten())
lr_rmse = mean_squared_error(y_test_actual.flatten(), lr_preds.flatten()) ** 0.5
lr_mape = safe_mape(y_test_actual, lr_preds, label="Linear Regression")

# ── 8. Baseline 4 — Random Forest (NEW) ────────────────────────────────────
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_sc, y_train_flat)
rf_preds = rf_model.predict(X_test_sc)

rf_mae = mean_absolute_error(y_test_actual.flatten(), rf_preds.flatten())
rf_rmse = mean_squared_error(y_test_actual.flatten(), rf_preds.flatten()) ** 0.5
rf_mape = safe_mape(y_test_actual, rf_preds, label="Random Forest")

print("\n✅ Baselines trained (Historical Average, Last Value, Linear Regression, Random Forest)")

# ── 9. MonsoonAwareSTGNN ────────────────────────────────────────────────────
class MonsoonAwareSTGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, monsoon_idx, modal_idx):
        super().__init__()
        self.monsoon_idx = monsoon_idx
        self.modal_idx = modal_idx
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.gru = nn.GRU(hidden_channels, hidden_channels, batch_first=True, num_layers=2)
        self.monsoon_attn = nn.Sequential(
            nn.Linear(len(monsoon_idx), hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels), nn.Sigmoid()
        )
        self.modal_attn = nn.Sequential(
            nn.Linear(len(modal_idx), hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels), nn.Sigmoid()
        )
        self.output = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x_seq, edge_index):
        T, N, Feat = x_seq.shape
        node_embeds = []
        for t in range(T):
            x = x_seq[t]
            monsoon_feat = x[:, self.monsoon_idx]
            monsoon_weight = self.monsoon_attn(monsoon_feat)
            modal_feat = x[:, self.modal_idx]
            modal_weight = self.modal_attn(modal_feat)
            h = F.relu(self.gcn1(x, edge_index))
            h = self.dropout(h)
            h = F.relu(self.gcn2(h, edge_index))
            h = h * monsoon_weight * modal_weight
            node_embeds.append(h)
        node_embeds = torch.stack(node_embeds, dim=1)  # [N, T, hidden]
        gru_out, _ = self.gru(node_embeds)              # [N, T, hidden]
        out = self.output(gru_out)                       # [N, T, 1]
        out = out.squeeze(-1).transpose(0, 1)             # [T, N] -- one prediction per snapshot, per node
        return out

model = MonsoonAwareSTGNN(
    in_channels=len(feature_cols), hidden_channels=64, out_channels=1,
    monsoon_idx=MONSOON_IDX, modal_idx=MODAL_IDX
).to(device)
optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)

train_losses = []
num_epochs = 100
print("\nTraining MonsoonAwareSTGNN...")

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train, edge_index_dev)
    loss = F.mse_loss(pred, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = model(X_test, edge_index_dev)
            test_pred_kmh = test_pred * (speed_max - speed_min) + speed_min
            y_test_kmh = y_test * (speed_max - speed_min) + speed_min
            mae = F.l1_loss(test_pred_kmh, y_test_kmh).item()
        print(f"Epoch {epoch+1:3d} | Train Loss: {loss.item():.4f} | MAE: {mae:.2f} km/h")

torch.save(model.state_dict(), models_dir / "monsoon_stgnn.pt")
print("✅ GNN training complete, model saved.")

# ── 10. GNN — per-snapshot evaluation (correct way) ────────────────────────
model.eval()
gnn_all_mae, gnn_all_rmse, gnn_all_mape = [], [], []
gnn_preds_by_snapshot = []

with torch.no_grad():
    for i in range(X_test.shape[0]):
        x_single = X_test[i:i+1]
        y_single = y_test[i]
        pred = model(x_single, edge_index_dev).squeeze(0)  # [1, N] -> [N]
        pred_kmh = pred * (speed_max - speed_min) + speed_min
        y_kmh = y_single * (speed_max - speed_min) + speed_min

        pred_kmh_np = pred_kmh.cpu().numpy()
        y_kmh_np = y_kmh.cpu().numpy()

        gnn_all_mae.append(F.l1_loss(pred_kmh, y_kmh).item())
        gnn_all_rmse.append(F.mse_loss(pred_kmh, y_kmh).item() ** 0.5)
        gnn_all_mape.append(safe_mape(y_kmh_np, pred_kmh_np))
        gnn_preds_by_snapshot.append(pred_kmh_np.tolist())

gnn_mae = float(np.mean(gnn_all_mae))
gnn_rmse = float(np.mean(gnn_all_rmse))
gnn_mape = float(np.nanmean(gnn_all_mape))

# ── 11. Final comparison ────────────────────────────────────────────────────
def m(v):
    """Cast numpy/torch scalars to native Python float, rounded to 2dp, JSON-safe."""
    return float(round(float(v), 2))

results_table = {
    "Historical Average": {"MAE": m(ha_mae), "RMSE": m(ha_rmse), "MAPE": m(ha_mape)},
    "Last Value":          {"MAE": m(lv_mae), "RMSE": m(lv_rmse), "MAPE": m(lv_mape)},
    "Linear Regression":   {"MAE": m(lr_mae), "RMSE": m(lr_rmse), "MAPE": m(lr_mape)},
    "Random Forest":       {"MAE": m(rf_mae), "RMSE": m(rf_rmse), "MAPE": m(rf_mape)},
    "MonsoonAwareSTGNN (Ours)": {"MAE": m(gnn_mae), "RMSE": m(gnn_rmse), "MAPE": m(gnn_mape)},
}

print("\n" + "=" * 70)
print(f"{'Model':<28} {'MAE (km/h)':>12} {'RMSE (km/h)':>13} {'MAPE (%)':>10}")
print("=" * 70)
for name, metrics in results_table.items():
    marker = "  <- OURS" if "Ours" in name else ""
    print(f"{name:<28} {metrics['MAE']:>12} {metrics['RMSE']:>13} {metrics['MAPE']:>10}{marker}")
print("=" * 70)

# ── 12. Last-snapshot predictions per junction (for dashboard) ────────────
last_snapshot_comparison = []
last_actual = y_test_actual[-1]
last_gnn_pred = gnn_preds_by_snapshot[-1]
last_rf_pred = rf_preds.reshape(y_test_actual.shape)[-1]
for i, jname in enumerate(junction_list):
    last_snapshot_comparison.append({
        "junction": jname,
        "actual_kmh": round(float(last_actual[i]), 1),
        "gnn_pred_kmh": round(float(last_gnn_pred[i]), 1),
        "rf_pred_kmh": round(float(last_rf_pred[i]), 1),
    })

# ── 13. Save everything for the dashboard ──────────────────────────────────
output = {
    "dataset_info": {
        "total_rows": int(merged_full.shape[0]),
        "snapshots": int(len(timestamps)),
        "junctions": int(num_nodes),
        "date_range_start": str(merged_full["timestamp"].min()),
        "date_range_end": str(merged_full["timestamp"].max()),
        "train_snapshots": int(split),
        "test_snapshots": int(len(timestamps) - split),
    },
    "results_table": results_table,
    "training_curve": [float(round(l, 4)) for l in train_losses],
    "last_snapshot_comparison": last_snapshot_comparison,
}

with open(logs_dir / "final_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"\n✅ Results saved to {logs_dir / 'final_results.json'}")
print("   Ready for dashboard generation (Step C).")

Loaded dataset: (3264, 34)
Snapshots: 165  |  Junctions: 20
Graph edges: 212
Feature tensor: (165, 20, 16)  |  Target tensor: (165, 20)
Train snapshots: 132  |  Test snapshots: 33
Device: cuda

⚠️ Data quality note: 32/660 test readings have actual speed < 5.0 km/h (likely failed API reads or stalled traffic). These are excluded from MAPE only, not from MAE/RMSE.
   ⚠️ Historical Average: excluded 32/660 rows with actual speed < 5.0 km/h from MAPE
   ⚠️ Last Value: excluded 32/660 rows with actual speed < 5.0 km/h from MAPE
   ⚠️ Linear Regression: excluded 32/660 rows with actual speed < 5.0 km/h from MAPE
   ⚠️ Random Forest: excluded 32/660 rows with actual speed < 5.0 km/h from MAPE

✅ Baselines trained (Historical Average, Last Value, Linear Regression, Random Forest)

Training MonsoonAwareSTGNN...
Epoch  20 | Train Loss: 0.0240 | MAE: 6.40 km/h
Epoch  40 | Train Loss: 0.0161 | MAE: 5.87 km/h
Epoch  60 | Train Loss: 0.0142 | MAE: 5.97 km/h
Epoch  80 | Train Loss: 0.0138 | MAE: 6.0

In [5]:
import json
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

logs_dir = Path("logs") if Path("logs").exists() else Path("../logs")

with open(logs_dir / "final_results.json") as f:
    data = json.load(f)

info = data["dataset_info"]
results = data["results_table"]
training_curve = data["training_curve"]
last_snap = data["last_snapshot_comparison"]

models = list(results.keys())
mae_vals = [results[m]["MAE"] for m in models]
rmse_vals = [results[m]["RMSE"] for m in models]
mape_vals = [results[m]["MAPE"] for m in models]

# Highlight our model differently
bar_colors = ["#4C78A8" if "Ours" not in m else "#E45756" for m in models]

junctions = [d["junction"] for d in last_snap]
actual = [d["actual_kmh"] for d in last_snap]
gnn_pred = [d["gnn_pred_kmh"] for d in last_snap]
rf_pred = [d["rf_pred_kmh"] for d in last_snap]

# ── Build dashboard ──────────────────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type": "table", "colspan": 2}, None],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "bar", "colspan": 2}, None],
    ],
    subplot_titles=(
        "Model Comparison — All Metrics",
        "MAE by Model (km/h, lower is better)",
        "MAPE by Model (%, lower is better)",
        f"Actual vs Predicted Speed — Last Test Snapshot ({info['junctions']} Junctions)",
    ),
    row_heights=[0.28, 0.36, 0.36],
    vertical_spacing=0.09,
)

# Row 1 — comparison table
fig.add_trace(
    go.Table(
        columnwidth=[220, 100, 100, 100],
        header=dict(
            values=["<b>Model</b>", "<b>MAE (km/h)</b>", "<b>RMSE (km/h)</b>", "<b>MAPE (%)</b>"],
            fill_color="royalblue", font=dict(color="white", size=13), align="center", height=36
        ),
        cells=dict(
            values=[models, mae_vals, rmse_vals, mape_vals],
            fill_color=[["lightyellow" if "Ours" not in m else "lightgreen" for m in models]] * 4,
            font=dict(size=12), align="center", height=32
        ),
    ),
    row=1, col=1
)

# Row 2 — MAE bar chart
fig.add_trace(
    go.Bar(x=models, y=mae_vals, marker_color=bar_colors, text=mae_vals, textposition="outside",
           name="MAE", showlegend=False),
    row=2, col=1
)

# Row 2 — MAPE bar chart
fig.add_trace(
    go.Bar(x=models, y=mape_vals, marker_color=bar_colors, text=mape_vals, textposition="outside",
           name="MAPE", showlegend=False),
    row=2, col=2
)

# Row 3 — Actual vs Predicted per junction (GNN + Random Forest)
fig.add_trace(
    go.Bar(x=junctions, y=actual, name="Actual", marker_color="#54A24B"),
    row=3, col=1
)
fig.add_trace(
    go.Bar(x=junctions, y=gnn_pred, name="GNN Predicted", marker_color="#E45756"),
    row=3, col=1
)
fig.add_trace(
    go.Bar(x=junctions, y=rf_pred, name="Random Forest Predicted", marker_color="#F58518"),
    row=3, col=1
)

fig.update_xaxes(tickangle=-45, row=2, col=1)
fig.update_xaxes(tickangle=-45, row=2, col=2)
fig.update_xaxes(tickangle=-45, row=3, col=1)
fig.update_yaxes(title_text="MAE (km/h)", row=2, col=1)
fig.update_yaxes(title_text="MAPE (%)", row=2, col=2)
fig.update_yaxes(title_text="Speed (km/h)", row=3, col=1)

fig.update_layout(
    height=1150,
    barmode="group",
    title=dict(
        text=(
            f"🚦 MonsoonAwareSTGNN — Chennai Traffic Prediction Dashboard<br>"
            f"<sub>{info['snapshots']} snapshots ({info['train_snapshots']} train / "
            f"{info['test_snapshots']} test) · {info['junctions']} junctions · "
            f"{info['date_range_start'][:10]} to {info['date_range_end'][:10]}</sub>"
        ),
        x=0.5, font=dict(size=20)
    ),
    legend=dict(orientation="h", yanchor="bottom", y=-0.12, xanchor="center", x=0.5),
    margin=dict(t=110, b=40),
)

out_path = logs_dir / "dashboard_final.html"
fig.write_html(out_path)
print(f"✅ Dashboard saved to {out_path}")

# ── Separate training curve chart ────────────────────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    y=training_curve, mode="lines", name="Train Loss (MSE)",
    line=dict(color="#4C78A8", width=2)
))
fig2.update_layout(
    title="MonsoonAwareSTGNN Training Curve",
    xaxis_title="Epoch", yaxis_title="MSE Loss (normalized)",
    height=400,
)
fig2.write_html(logs_dir / "training_curve.html")
print(f"✅ Training curve saved to {logs_dir / 'training_curve.html'}")

print("\nOpen both HTML files in your browser to view the dashboard.")

✅ Dashboard saved to ..\logs\dashboard_final.html
✅ Training curve saved to ..\logs\training_curve.html

Open both HTML files in your browser to view the dashboard.


In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pathlib import Path

raw_dir = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")
logs_dir = Path("logs") if Path("logs").exists() else Path("../logs")
logs_dir.mkdir(parents=True, exist_ok=True)

# ── Reload latest data ─────────────────────────────────────────────────────
df = pd.read_csv(raw_dir / "chennai_traffic_log.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")

print(f"Loaded {len(df)} rows")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Unique snapshots: {df['timestamp'].nunique()}")

# ── Dashboard Layout ────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Live Speed by Junction',
        'Speed Over Time — Top 5 Junctions',
        'Junction Map — Chennai',
        'Traffic Congestion Index'
    ),
    specs=[
        [{"type": "bar"}, {"type": "scatter"}],
        [{"type": "scattermapbox"}, {"type": "bar"}]
    ]
)

# ── Plot 1: Latest speed per junction ──────────────────────────────────────
latest = df.groupby('junction').last().reset_index()
colors = ['red' if s < 15 else 'orange' if s < 25 else 'green'
          for s in latest['current_speed']]

fig.add_trace(go.Bar(
    x=latest['junction'], y=latest['current_speed'], marker_color=colors,
    name='Current Speed', text=latest['current_speed'], textposition='outside'
), row=1, col=1)

# ── Plot 2: Speed over time for top 5 busiest junctions ───────────────────
busy_junctions = ['Kathipara', 'T_Nagar_Panagal', 'Koyambedu',
                   'Anna_Nagar_Roundtana', 'Adyar_Signal']
colors_line = ['blue', 'red', 'green', 'purple', 'orange']

for junc, col in zip(busy_junctions, colors_line):
    jdf = df[df['junction'] == junc].sort_values('timestamp')
    fig.add_trace(go.Scatter(
        x=jdf['timestamp'], y=jdf['current_speed'], mode='lines+markers',
        name=junc, line=dict(color=col, width=2), marker=dict(size=4)
    ), row=1, col=2)

# ── Plot 3: Map of junctions ───────────────────────────────────────────────
fig.add_trace(go.Scattermapbox(
    lat=latest['lat'], lon=latest['lon'], mode='markers+text',
    marker=dict(size=15, color=latest['current_speed'], colorscale='RdYlGn',
                cmin=0, cmax=60, showscale=True, colorbar=dict(title='km/h', x=0.48)),
    text=latest['junction'], textposition='top right',
    hovertext=[f"{j}: {s} km/h" for j, s in zip(latest['junction'], latest['current_speed'])],
    name='Junctions'
), row=2, col=1)

# ── Plot 4: Congestion Index (free_flow vs current) ────────────────────────
latest['congestion'] = ((latest['free_flow_speed'] - latest['current_speed'])
                         / latest['free_flow_speed'] * 100).clip(0, 100)
cong_colors = ['red' if c > 40 else 'orange' if c > 20 else 'green'
               for c in latest['congestion']]

fig.add_trace(go.Bar(
    x=latest['junction'], y=latest['congestion'], marker_color=cong_colors,
    name='Congestion %', text=[f"{c:.0f}%" for c in latest['congestion']], textposition='outside'
), row=2, col=2)

# ── Layout ───────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(f'🚦 MonsoonAwareSTGNN — Chennai Traffic Dashboard '
              f'(Live data through {df["timestamp"].max().strftime("%b %d, %Y")})'),
        font=dict(size=18)
    ),
    height=900,
    mapbox=dict(style='carto-positron', center=dict(lat=13.0, lon=80.2), zoom=10),
    showlegend=True,
    template='plotly_white'
)
fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)

fig.write_html(logs_dir / "dashboard.html")
print(f"\n✅ Dashboard refreshed and saved to {logs_dir / 'dashboard.html'}")

Loaded 3264 rows
Date range: 2026-07-10 21:43:48.458304 to 2026-07-25 09:20:29.732710
Unique snapshots: 165


C:\Users\itsta\AppData\Local\Temp\ipykernel_22212\791629814.py:56: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(



✅ Dashboard refreshed and saved to ..\logs\dashboard.html
